### 1. Importing Libraries

In [1]:
import os
import cv2 as cv
import numpy as np

import matplotlib.pyplot as plt

from keras.models import Model, load_model
from keras.layers import Input, BatchNormalization, Activation, Dense, Dropout
from keras.layers import Conv2D, Conv2DTranspose
from keras.layers import MaxPooling2D, GlobalMaxPool2D
from keras.layers import concatenate
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.optimizers import Adam
import tensorflow as tf

### 2. Model Definition (UNet 64x256)

In [2]:
def unet_64to256(input_shape=(64, 64, 3), n_classes=3, final_activation='sigmoid', dropout_rate=0.05):
    inputs = Input(shape=input_shape, name='img')

    # -------------------------
    # Encoder
    # -------------------------
    c1 = Conv2D(16, (3,3), padding='same')(inputs)
    c1 = BatchNormalization()(c1); c1 = Activation('relu')(c1)
    c1 = Conv2D(16, (3,3), padding='same')(c1)
    c1 = BatchNormalization()(c1); c1 = Activation('relu')(c1)
    p1 = MaxPooling2D((2,2))(c1); p1 = Dropout(dropout_rate)(p1)   # 64 -> 32

    c2 = Conv2D(32, (3,3), padding='same')(p1)
    c2 = BatchNormalization()(c2); c2 = Activation('relu')(c2)
    c2 = Conv2D(32, (3,3), padding='same')(c2)
    c2 = BatchNormalization()(c2); c2 = Activation('relu')(c2)
    p2 = MaxPooling2D((2,2))(c2); p2 = Dropout(dropout_rate)(p2)   # 32 -> 16

    c3 = Conv2D(64, (3,3), padding='same')(p2)
    c3 = BatchNormalization()(c3); c3 = Activation('relu')(c3)
    c3 = Conv2D(64, (3,3), padding='same')(c3)
    c3 = BatchNormalization()(c3); c3 = Activation('relu')(c3)
    p3 = MaxPooling2D((2,2))(c3); p3 = Dropout(dropout_rate)(p3)   # 16 -> 8

    c4 = Conv2D(128, (3,3), padding='same')(p3)
    c4 = BatchNormalization()(c4); c4 = Activation('relu')(c4)
    c4 = Conv2D(128, (3,3), padding='same')(c4)
    c4 = BatchNormalization()(c4); c4 = Activation('relu')(c4)
    p4 = MaxPooling2D((2,2))(c4); p4 = Dropout(dropout_rate)(p4)   # 8 -> 4

    # -------------------------
    # Bottleneck
    # -------------------------
    c5 = Conv2D(256, (3,3), padding='same')(p4)
    c5 = BatchNormalization()(c5); c5 = Activation('relu')(c5)
    c5 = Conv2D(256, (3,3), padding='same')(c5)
    c5 = BatchNormalization()(c5); c5 = Activation('relu')(c5)

    # -------------------------
    # Decoder
    # -------------------------
    u6 = Conv2DTranspose(128, (3,3), strides=(2,2), padding='same')(c5)
    u6 = concatenate([u6, c4]); u6 = Dropout(dropout_rate)(u6)
    u6 = Conv2D(128, (3,3), padding='same')(u6)
    u6 = BatchNormalization()(u6); u6 = Activation('relu')(u6)
    u6 = Conv2D(128, (3,3), padding='same')(u6)
    u6 = BatchNormalization()(u6); u6 = Activation('relu')(u6)

    u7 = Conv2DTranspose(64, (3,3), strides=(2,2), padding='same')(u6)
    u7 = concatenate([u7, c3]); u7 = Dropout(dropout_rate)(u7)
    u7 = Conv2D(64, (3,3), padding='same')(u7)
    u7 = BatchNormalization()(u7); u7 = Activation('relu')(u7)
    u7 = Conv2D(64, (3,3), padding='same')(u7)
    u7 = BatchNormalization()(u7); u7 = Activation('relu')(u7)

    u8 = Conv2DTranspose(32, (3,3), strides=(2,2), padding='same')(u7)
    u8 = concatenate([u8, c2]); u8 = Dropout(dropout_rate)(u8)
    u8 = Conv2D(32, (3,3), padding='same')(u8)
    u8 = BatchNormalization()(u8); u8 = Activation('relu')(u8)
    u8 = Conv2D(32, (3,3), padding='same')(u8)
    u8 = BatchNormalization()(u8); u8 = Activation('relu')(u8)

    u9 = Conv2DTranspose(16, (3,3), strides=(2,2), padding='same')(u8)
    u9 = concatenate([u9, c1]); u9 = Dropout(dropout_rate)(u9)
    u9 = Conv2D(16, (3,3), padding='same')(u9)
    u9 = BatchNormalization()(u9); u9 = Activation('relu')(u9)
    u9 = Conv2D(16, (3,3), padding='same')(u9)
    u9 = BatchNormalization()(u9); u9 = Activation('relu')(u9)

    # Upsample 64→128
    u10 = Conv2DTranspose(16, (3,3), strides=(2,2), padding='same')(u9)
    u10 = Dropout(dropout_rate)(u10)
    u10 = Conv2D(16, (3,3), padding='same')(u10)
    u10 = BatchNormalization()(u10); u10 = Activation('relu')(u10)

    # Upsample 128→256 (no skip)
    u11 = Conv2DTranspose(16, (3,3), strides=(2,2), padding='same')(u10)
    u11 = Dropout(dropout_rate)(u11)
    u11 = Conv2D(16, (3,3), padding='same')(u11)
    u11 = BatchNormalization()(u11); u11 = Activation('relu')(u11)

    outputs = Conv2D(n_classes, (1,1), activation=final_activation, name='mask')(u11)
    return Model(inputs=inputs, outputs=outputs, name='UNet_64to256')

### 3. Model Initialization and Compilation

In [3]:
model = unet_64to256(input_shape=(64,64,3), n_classes=3, final_activation='sigmoid')

print('Input shape:', model.input_shape)   # (None, 64, 64, 3)
print('Output shape:', model.output_shape) # (None, 256, 256, 3)

model.compile(optimizer=Adam(1e-4), loss='mae' )

Input shape: (None, 64, 64, 3)
Output shape: (None, 256, 256, 3)


### 4. Downloading Kaggale dataset

In [4]:
import json
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = "rahuldev110"
os.environ['KAGGLE_KEY'] = userdata.get("KAGGLE_API_KEY")


kaggle_json = {
    "username": os.environ['KAGGLE_USERNAME'],
    "key": os.environ['KAGGLE_KEY']
}

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_json, f)

os.chmod("/root/.kaggle/kaggle.json", 0o600)


In [6]:
!mkdir -p /content/celeba
!kaggle datasets download -d jessicali9530/celeba-dataset -p /content/celeba --unzip

Dataset URL: https://www.kaggle.com/datasets/jessicali9530/celeba-dataset
License(s): other
100% 1.33G/1.33G [00:12<00:00, 146MB/s]
100% 1.33G/1.33G [00:12<00:00, 119MB/s]


### 5. Data Generator

In [13]:
import os
import cv2 as cv
import numpy as np

DATA_DIR = '/content/celeba/img_align_celeba/img_align_celeba/'
imgs = os.listdir(DATA_DIR)

def datagen(batch_size):
    while True:
        x_batch, y_batch = [], []
        while len(x_batch) < batch_size:
            idx = np.random.randint(0, len(imgs))
            img_path = os.path.join(DATA_DIR, imgs[idx])
            bgr = cv.imread(img_path)
            if bgr is None:
                continue
            rgb = cv.cvtColor(bgr, cv.COLOR_BGR2RGB)
            x = cv.resize(rgb, (64, 64)) / 255.0
            y = cv.resize(rgb, (256, 256)) / 255.0
            x_batch.append(x)
            y_batch.append(y)
        yield np.array(x_batch, dtype=np.float32), np.array(y_batch, dtype=np.float32)



### 6. Model Training

In [40]:
import os, cv2 as cv, numpy as np, matplotlib.pyplot as plt, tensorflow as tf

def show_and_save_samples(model, imgs, base_dir, epoch, n=5, save_dir='./epoch_samples'):
    os.makedirs(save_dir, exist_ok=True)
    chosen = np.random.choice(imgs, n, replace=False)
    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
    if n == 1: axes = np.expand_dims(axes, 0)

    for i, fname in enumerate(chosen):
        path = os.path.join(base_dir, fname)
        bgr = cv.imread(path)
        if bgr is None:
            continue
        rgb = cv.cvtColor(bgr, cv.COLOR_BGR2RGB)

        # Model input: 64x64
        img64 = cv.resize(rgb, (64,64)).astype('float32') / 255.0
        # Ground truth: 256x256
        gt256 = cv.resize(rgb, (256,256)).astype('float32') / 255.0

        # Model prediction
        pred = model.predict(np.expand_dims(img64,0), verbose=0)[0]

        # Upsample input for visualization
        lowup = cv.resize((img64*255).astype(np.uint8), (256,256))
        pred_vis = (pred*255).astype(np.uint8)

        for ax, im, title in zip(axes[i], [rgb, gt256, lowup, pred_vis],
                                 ['Original','GT 256x256','Low-Res Upsampled','Prediction']):
            ax.imshow(im)
            ax.set_title(title)
            ax.axis('off')

    plt.tight_layout()
    plt.savefig(f"{save_dir}/epoch_{epoch+1:02d}.png")
    plt.show()


# --- Callback ---
base_dir = '/content/celeba/img_align_celeba/img_align_celeba/'  # adjust path if needed
imgs = [f for f in os.listdir(base_dir) if f.lower().endswith(('.jpg', '.png'))]
os.makedirs('./checkpoints', exist_ok=True)

def on_epoch_end(epoch, logs):
    show_and_save_samples(model, imgs, base_dir, epoch, n=5)
    model.save(f'./checkpoints/model_epoch_{epoch+1:02d}.keras')

show_cb = tf.keras.callbacks.LambdaCallback(on_epoch_end=on_epoch_end)


# --- Training ---
batch_size = 32
steps_per_epoch = len(imgs) // batch_size

results = model.fit(
    datagen(batch_size=batch_size),
    steps_per_epoch=steps_per_epoch,
    epochs=3,
    callbacks=[show_cb],
    verbose=1
)


Output hidden; open in https://colab.research.google.com to view.